# One-armed bandit

Here we look at a one-armed bandit problem with finite horizon $T$. This is a bandit problem where there are 2 actions:

Action 1: Play
Action 0: Stop.

If you stop, you get a reward of 0.5 until the end of the game.

If you play, then you get a reward drawn from an unknown distribution with parameter $\theta$, i.e.
$r_t \mid a_t = 1, \theta \sim P_\theta$

If $E_P[r_t] > 0$ then the optimal choice is to play action 1, otherwise action 0. 
However, we do not know $P$.


# Warm-up

In this setting, we assume that the unknown bandit arm has two possible reward distributions: $P_1$ is Bernoulli($\theta = 0.25$), and $P_2$ is Bernoulli($\theta = 0.75$). We use a Bayesian setting, and have a prior over the two Bernoulli possibilities:

$p = \xi(\theta = 0.25)$, and $1 - p = \xi(\theta = 0.75)$

Now, if we play Stop, then we never learn anything about the bandit. If we Play, then we obtain a reward. No matter what reward we obtain, we can calculate the posterior distribution

$\xi(\theta | r_t, a_t ) = P(r_t \mid a_t, \theta) \xi(\theta) / P_\xi(r_t \mid a_t = 1)$.

Here, the denominator is the marginal distribution
$P_\xi(r_t \mid a_t = 1) = \sum_{\theta \in \{0.25, 0.75\}} P(r_t \mid a_t, \theta) \xi(\theta)$.

We can always calculate the posterior recursively. If we set $\xi_0(\theta) = \xi(\theta)$ and $\xi_t(\theta) = \xi(\theta | r_1, \ldots r_t, a_1, \ldots, a_t)$ then

$\xi_{t+1}(\theta) = P(r_{t+1} \mid a_{t+1}, \theta) \xi_t(\theta) / P_{\xi_t}(r_{t+1} \mid a_{t+1})$.


In [1]:
# Posterior calculation

# What is the likelihood - assume it is a Bernoulli-distributed reward
# Input:
# - reward: what we obtained at this time step, taking values in {0,1}
# - theta: the parameter we have, taking values in [0,1]
# Output:
# - return the probability of this reward given theta $P(r | \theta)$
def get_likelihood(reward, theta):
    return np.pow(theta, reward) * np.pow(1 - theta, 1 - reward)

# What is the posterior
# Input:
# - reward: what we obtained at this time step
# - prior: the discrete prior over theta, so prior[i] is the probability of theta[i]
# - Theta: the set of theta parameters Theta[i]
# Output:
# - return the probability of theta given the reward, posterior[i]
def get_posterior(prior, Theta, reward):
    n_theta = len(Theta)
    L = np.zeros(n_theta)
    for i in range(n_theta):
        L[i] = get_likelihood(reward, Theta[i]) # the probability of the reward for each possible theta
    posterior = prior * L # same size prior and L gives me a vector of same size, element-wise mul
    marginal = sum(posterior)
    posterior /= marginal                      
    return posterior



# Backwards induction in belief space

What should we do if we only have action to play? What about two?
If it is two, then we can consider the options: Play, or Stop.
If we Play, we can then Play or Stop int he second move.

However, if we play, what is the probability that we observe a reward? That is given by the marignal distribution.


In [2]:
# What is the marginal?
# In this function, we only care about action 1, since action 0 ends the game.

# Input:
# - reward: the reward we might observe if we play action 1
# - prior: the discrete prior over theta, so prior[i] is the probability of theta[i]
# - Theta: the set of theta parameters theta[i],
# Output:
# - return the probability of that reward
def get_marginal(reward, prior, Theta):
    n_theta = len(Theta)
    L = np.zeros(n_theta)
    for i in range(n_theta):
        L[i] = prior[i] * get_likelihood(reward, Theta[i]) # the probability of the reward for each possible theta
    marginal = sum(L)
    return marginal


# Constructing the Belief MDP.

We now think of the belief $\xi_t$ as the state.
We need to construct the transition distribution. Here, the transition only depends on the reward observed. In fact, the next state is just the posterior: so, it is a function of the prior probability, the reward and the action.

Note that for this example, the expected value of the reward is the probability of getting a reward of 1.

The Bellman equation is just like in a normal MDP, but the state is our _belief_ - the environment does not change.

$V(\xi_t) = \max_a E[r_t | a_t = a, \xi_t] + \sum_{\xi_{t+1}} P(\xi_{t+1} | \xi_t, a_t = a) V(\xi_{t+1})$

So for the first term, we already know:
$E[r_t | a_t = 0, \xi_t]$ is just a fixed reward of $(T -t +1)/2$.

$E[r_t | a_t = 1, \xi_t] = \sum_\theta \theta \xi_t(\theta)$, i.e. the marginal

Since $\xi_{t+1}$ is a function $r_t, \xi_t$, we only need to look at the probabilities of $r_t$, i.e. $\xi_{t+1}(\theta) = \xi_{t}(\theta \mid r_t, a_t)$. So, instead of summing over all possible next $\xi$, we sum over the possibile beliefs. In math terms,

$\sum_{\xi_{t+1}} P(\xi_{t+1} | \xi_t, a_t = a) V(\xi_{t+1}) = 
\sum_{r_{t}} P(r_t | \xi_t, a_t = a) V(\xi_{t}(\cdot | r_t, a_t))$.

The code below implements this, with get_posterior() effectively calculating the next belief state.


In [1]:
# Input:
# - prior: the prior over Thetas
# - Theta: the set of Thetas
# - horizon: How much we have left 
# Output:
# Return utility
# - utility: total reward from that point if playing optimally
def backwards_induction(prior, Theta, horizon):
    Q = np.zeros(2) # store the values here for each action
    Q[0] = 0.5 * horizon # for action 0, we just get 0.5 until the end
    
    # How likely is r = 1?
    P_r_is_one = get_marginal(1, prior, Theta) # also equal to expected reward IN THIS CASE

    # this is the expected reward
    Q[1] = P_r_is_one 
    # So, now there are two possibilities: we get a reward of 1
    Q[1] += P_r_is_one * backwards_induction(get_posterior(prior, Theta, 1), Theta, horizon -1)
    # or a reward of zero
    Q[1] += (1 - P_r_is_one) * backwards_induction(get_posterior(prior, Theta, 0), Theta, horizon-1)
    utility = max(Q)
    return utility